In [ ]:
"""STEP 1: Nazia's Rules tab  ->  rules/footwear_rules.json

Reads her existing "Rules" tab directly:
  column A  tariff code              column E  excluded inputs        column H  value threshold
  column D  tariff shift allowed     column F  value test (Yes/No)    column I  de minimis
  column J  US general duty          column K  Chapter 99 notes       column L  Sec. 232/338 notes
and pulls the numbers out of her text (e.g. "Duty line: 6401.99.10", "+10%", "YES ... +50%").

Run:  python step1_rules.py Nazia_Rules_Workbook.xlsx

"""

from google.colab import drive
drive.mount('/content/drive')


import json, pathlib, re, sys
import openpyxl

# ---- FOLDERS ----
DRIVE    = pathlib.Path('/content/drive/MyDrive/Deep_Learning')
WORKBOOK = DRIVE / 'Rules' / 'Nazia_Rules_Workbook.xlsx'   # reads Nazia's Rules tab
OUT      = DRIVE / 'Rules' / 'footwear_rules.json'         # writes the rules file


# Settings that are NOT on her Rules tab: team decisions, kept here in one place.
TEAM_SETTINGS = {
    "escalation_margin": 0.02,          # within 2 points of the threshold -> a person decides
    "knee_review_band_in": [14, 17],    # Assumption #8: boots 14-17 in. tall -> a person decides
}


def pct(text, default=None):
    m = re.search(r"(\d+(?:\.\d+)?)\s*%", str(text))
    return float(m.group(1)) / 100 if m else default


def main(path):
    ws = openpyxl.load_workbook(path, data_only=True)["Rules"]
    header_row = next(r for r in range(1, 15) if str(ws.cell(r, 1).value or "").startswith("Tariff code"))
    footwear, duty, used = None, {}, []

    for r in range(header_row + 1, ws.max_row + 1):
        code6 = ws.cell(r, 1).value
        if code6 in (None, ""):
            continue
        shift, excluded, value_test = ws.cell(r, 4).value, ws.cell(r, 5).value, ws.cell(r, 6).value
        threshold, de_min, general = ws.cell(r, 8).value, ws.cell(r, 9).value, ws.cell(r, 10).value
        ch99, special = str(ws.cell(r, 11).value or ""), str(ws.cell(r, 12).value or "")

        # The US 8-digit duty line is written in column K ("Duty line: 6401.99.10 ...")
        m = re.search(r"(\d{4}\.\d{2}\.\d{2})", ch99)
        if not m:
            print(f"  row {r}: no 8-digit duty line found in column K, skipped")
            continue
        code8 = m.group(1)
        extra = pct(re.search(r"\+\s*\d+(?:\.\d+)?\s*%", ch99).group(0)) if re.search(r"\+\s*\d+(?:\.\d+)?\s*%", ch99) else 0.0
        anyway = pct(special, 0.0) if special.strip().upper().startswith("YES") else 0.0
        scope = "stretch" if str(value_test).strip().lower() == "no" else "core"
        duty[code8] = {"general": float(general), "usmca": 0.0, "extra_if_not_qualifying": extra,
                       "applies_anyway": anyway, "scope": scope, "from_rules_row": r}
        used.append((r, code6, code8))

        # A second line in her notes, e.g. "Exception: 6401.92.60 (>90% PVC ...) 4.6%"
        for m2 in re.finditer(r"Exception:\s*(\d{4}\.\d{2}\.\d{2})\s*\([^)]*\)\s*(\d+(?:\.\d+)?)%", ch99):
            duty[m2.group(1)] = dict(duty[code8], general=float(m2.group(2)) / 100)
            used.append((r, code6, m2.group(1)))

        # The footwear rule itself comes from the footwear rows (value test = Yes)
        if scope == "core" and footwear is None:
            rng = re.search(r"(\d{2})\.(\d{2})\s*-\s*(\d{2})\.(\d{2})", str(shift))
            footwear = {
                "excluded_headings_range": [int(rng.group(1) + rng.group(2)), int(rng.group(3) + rng.group(4))],
                "excluded_subheadings": sorted(set(re.findall(r"\b\d{4}\.\d{2}\b", str(excluded)))),
                "rvc_method": "net_cost",
                "rvc_threshold": float(threshold),
                "de_minimis_share": pct(de_min),
            }

    if footwear is None or not duty:
        sys.exit("Could not find the footwear rows on the Rules tab; nothing written.")
    lo, hi = footwear["excluded_headings_range"]
    footwear["applies_to_headings"] = [str(h) for h in range(lo, hi + 1)]
    footwear["escalation_margin"] = TEAM_SETTINGS["escalation_margin"]
    rules = {"rules_version": str(ws.cell(header_row + 1, 14).value or "")[:10],
             "source": f"{pathlib.Path(path).name}, Rules tab (read by step1_rules.py)",
             "footwear_rule": footwear,
             "classification": {"knee_review_band_in": TEAM_SETTINGS["knee_review_band_in"]},
             "us_duty": duty}
    OUT.parent.mkdir(exist_ok=True)
    OUT.write_text(json.dumps(rules, indent=2))

    print(f"Read {path} -> wrote {OUT}")
    print("The footwear rule:")
    for k, v in footwear.items():
        print(f"   {k:24} {v}")
    print("Duty rates (from Rules tab row -> code):")
    for r, c6, c8 in used:
        d = duty[c8]
        print(f"   row {r:>2}  {str(c6):8} -> {c8}  general {d['general']:.1%}  +{d['extra_if_not_qualifying']:.0%} if not qualifying"
              f"  {d['applies_anyway']:.0%} anyway  ({d['scope']})")


main(WORKBOOK)
